# Phase 1 - First Look & Data Quality Report

**Goal:** profile the raw Zomato India dataset *before* cleaning anything. We only
observe and document problems here. The output of this notebook is a **Data
Quality Report** - the list of issues our cleaning pipeline (Phase 2) will fix.

Rule: the raw CSV in `../data/` is **immutable**. We never overwrite it.

In [1]:
import pandas as pd
import numpy as np

# low_memory=False because the cost column has mixed types (numbers + '1,200').
RAW = "../data/india_all_restaurants_details.csv"
df = pd.read_csv(RAW, low_memory=False)

print("Rows:", df.shape[0], "| Columns:", df.shape[1])
df.head(5)

Rows: 224520 | Columns: 18


,Unnamed: 0,sno,zomato_url,name,city,area,rating,rating_count,telephone,cusine,cost_for_two,address,coordinates,timings,online_order,table_reservation,delivery_only,famous_food
0,0,0,https://www.zomato.com/ncr/sainik-food-pandav-...,Sainik Food,Delhi NCR,Pandav Nagar,3.2,21,011 22486474 +91 9717806814,North Indian,300,"C 4/1, Opposite Mother Dairy, Pandav Nagar, Ne...","28.6177324058,77.2848711535","{'Mon': '1pm – 4pm, 7pm – 10:30pm', 'Tue': '1p...",False,False,False,NaN
1,1,1,https://www.zomato.com/mumbai/kunals-creamery-...,Kunal's Creamery & Eatery,Mumbai,Ambernath,3.6,51,+91 9561356690 +91 9637537499,"Street Food, Chinese, Fast Food",500,"Shop 14, Trishul Shivdham Complex, Shiv Mandir...","19.2058869331,73.1842865422","{'Mon': '10am – 12midnight', 'Tue': '10am – 12...",False,False,False,Penne Pasta
2,2,2,https://www.zomato.com/ncr/brij-palace-restaur...,Brij Palace Restaurant,Delhi NCR,Jasola,0,0,+91 9891828106,North Indian,250,"4, Okhla Bus Stand, Jamia Nagar, Near, Jasola,...","28.5630343606,77.2912229598","{'Mon': '12noon – 4pm, 7pm – 12midnight', 'Tue...",False,False,False,"Chana Masala , Butter Naan , Paneer Bhujia , D..."
3,3,3,https://www.zomato.com/ncr/sahib-hotel-paharga...,Sahib Hotel,Delhi NCR,Paharganj,0,0,+91 9670005455,North Indian,300,"121, Amrit Kaur Market, Opposite New Delhi Rai...","28.6424106380,77.2182980552","{'Mon': '6am – 11:30pm, 12midnight – 1am', 'Tu...",False,False,False,NaN
4,4,4,https://www.zomato.com/kolkata/chunkys-shibpur...,Chunky's,Kolkata,Shibpur,3.0,78,+91 8442828284,"Italian, Pizza, Continental",500,"523, G.T Road, Howrah., Shibpur, Howrah","22.5777582163,88.3307084441","{'Mon': '12noon – 3am', 'Tue': '12noon – 3am',...",True,False,False,NaN


In [2]:
# Column names and data types - the first thing to check.
print(df.dtypes)

Unnamed: 0            int64
sno                   int64
zomato_url           object
name                 object
city                 object
area                 object
rating               object
rating_count          int64
telephone            object
cusine               object
cost_for_two         object
address              object
coordinates          object
timings              object
online_order           bool
table_reservation      bool
delivery_only          bool
famous_food          object
dtype: object


In [3]:
# Completeness check: how many missing values per column?
print(df.isnull().sum().sort_values(ascending=False))

famous_food          171994
timings                2964
address                1786
sno                       0
delivery_only             0
table_reservation         0
online_order              0
coordinates               0
cost_for_two              0
Unnamed: 0                0
telephone                 0
rating_count              0
rating                    0
area                      0
city                      0
name                      0
zomato_url                0
cusine                    0
dtype: int64


In [4]:
# Key column distributions - look for messy / non-numeric values.
print("--- rating (top 15) ---")
print(df["rating"].value_counts(dropna=False).head(15))
print("\n--- cost_for_two: how many have commas? ---")
print(df["cost_for_two"].astype(str).str.contains(",").sum(), "values contain a comma")
print("\n--- city (top 15) ---")
print(df["city"].value_counts().head(15))
print("\n--- cusine sample ---")
print(df["cusine"].head(5).tolist())

--- rating (top 15) ---
rating
0      52052
NEW    27731
3.3    12991
3.4    12901
3.5    12578
3.2    12220
3.6    12206
3.7    11411
3.8    10194
3.1     9650
3.9     8647
3.0     5939
2.9     5604
4.1     4825
4.0     4125
Name: count, dtype: int64

--- cost_for_two: how many have commas? ---


10013 values contain a comma

--- city (top 15) ---


city
Delhi NCR     38699
Mumbai        25692
Bengaluru     20283
Pune          15430
Hyderabad     12393
Chennai       11917
Kolkata        9571
Ahmedabad      6432
Jaipur         5367
Chandigarh     4278
Lucknow        3822
Goa            3572
Indore         3085
Nagpur         2954
Kochi          2685
Name: count, dtype: int64

--- cusine sample ---
['North Indian', 'Street Food, Chinese, Fast Food', 'North Indian', 'North Indian', 'Italian, Pizza, Continental']


In [5]:
# Uniqueness check: exact duplicate rows, and logical duplicates.
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate (name, city, area):",
      df.duplicated(subset=["name", "city", "area"]).sum())

Exact duplicate rows: 0
Duplicate (name, city, area): 9152


## Data Quality Report (what we found)

Documented **before** writing any cleaning code, so the pipeline is designed with intent:

| # | Issue | Column | Fix planned (Phase 2) |
|---|-------|--------|-----------------------|
| 1 | `rating` stored as text with `'NEW'`, `'0'`, and garbage (`'Nov...'`) | `rating` | Map non-numeric -> NaN, cast to float |
| 2 | `'0'` / `'NEW'` mean *unrated* (new restaurant), not a real 0 score | `rating` | Treat as NaN, not zero |
| 3 | `cost_for_two` has thousands separators (`'1,200'`) so it's text, not numeric | `cost_for_two` | Strip commas, cast to float |
| 4 | Cost of `0` is invalid (not "free") | `cost_for_two` | Map `<= 0` -> NaN |
| 5 | `cusine` packs up to 8 cuisines in one cell, comma-separated | `cusine` | Split + explode into a cuisine table |
| 6 | ~9,000 logical duplicates on (name, city, area) | all | Note; keep unless exact dup |
| 7 | `famous_food` ~77% null | `famous_food` | Drop from analysis set |

**Completeness:** ~35% of restaurants are unrated (`NEW`/`0`) - this is a *finding*, not
corruption: the market has a large tail of brand-new restaurants. **Structural** quality
(valid cost + city) is ~98%.